# Langkah 1: Ekspor Citra Satelit dari Google Earth Engine
Notebook ini digunakan untuk mengambil data citra Sentinel-1 (Radar) dan Sentinel-2 (Optik) berdasarkan titik alert deforestasi.

In [ ]:
import ee

# Otentikasi dan Inisialisasi ke Earth Engine
ee.Authenticate()
ee.Initialize()

In [ ]:
# Menentukan Titik Koordinat (ROI)
# Titik contoh (ganti dengan koordinat alert Anda)
lat, lon = -1.5, 113.0
point = ee.Geometry.Point([lon, lat])
roi = point.buffer(500).bounds()  # area 1 km x 1 km

start = '2023-06-01'
end   = '2023-07-01'

In [ ]:
# Sentinel-2 (Optik): Mengambil 4 band reflektansi dengan cloud cover rendah
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(roi).filterDate(start, end)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .median()
      .select(['B2','B3','B4','B8'])
      .rename(['S2_B2','S2_B3','S2_B4','S2_B8']))

# Sentinel-1 (Radar): Menembus awan, mengambil band VV dan VH
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(roi).filterDate(start, end)
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .median()
      .select(['VV','VH'])
      .rename(['S1_VV','S1_VH']))

# Menggabungkan Optik dan Radar menjadi 6-band image
stack = s2.addBands(s1).clip(roi)

In [ ]:
# Mengekspor ke Google Drive sebagai GeoTIFF
task = ee.batch.Export.image.toDrive(
    image=stack,
    description='alert_stack_kalimantan',
    scale=10,
    region=roi,
    fileFormat='GeoTIFF'
)
task.start()
print('Tugas ekspor ke Drive telah dimulai. Silakan periksa Google Drive Anda beberapa saat lagi.')